# Passing Quality — Final Model

**Naive**: `from_Passing_quality` only

**Tactical**: `from_Passing_quality` + delta ATTACK + delta DEFENCE + delta OUTCOME

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
DATA_DIR = "../../../../thesis_data/processed_data/thesis_model_dataset/active/"
df = pd.read_parquet(DATA_DIR + "within_league_transfers_v5.parquet")
mf = df[df["from_position"] == "Midfielder"].copy()

mf["delta_PQ"] = mf["to_Passing quality"] - mf["from_Passing quality"]
for tq in ["ATTACK", "DEFENCE", "OUTCOME"]:
    mf[f"delta_tq_{tq}"] = mf[f"to_q_{tq}"] - mf[f"from_q_proj_{tq}"]

delta_tq = ["delta_tq_ATTACK", "delta_tq_DEFENCE", "delta_tq_OUTCOME"]
mf = mf.dropna(subset=["from_Passing quality", "delta_PQ"] + delta_tq)

train, test = train_test_split(mf, test_size=0.2, random_state=42)
y_train = train["delta_PQ"]
y_test  = test["delta_PQ"]
print(f"Train: {len(train):,}  |  Test: {len(test):,}")

---
## Naive Model

In [ ]:
X_tr = sm.add_constant(train[["from_Passing quality"]])
X_te = sm.add_constant(test[["from_Passing quality"]])

naive = sm.OLS(y_train, X_tr).fit()
naive_pred = naive.predict(X_te)

print(naive.summary())
print(f"\nTest R²: {r2_score(y_test, naive_pred):.4f}  |  MAE: {mean_absolute_error(y_test, naive_pred):.4f}")

---
## Tactical Model

In [ ]:
feat = ["from_Passing quality"] + delta_tq
X_tr = sm.add_constant(train[feat])
X_te = sm.add_constant(test[feat])

tactical = sm.OLS(y_train, X_tr).fit()
tactical_pred = tactical.predict(X_te)

print(tactical.summary())
print(f"\nTest R²: {r2_score(y_test, tactical_pred):.4f}  |  MAE: {mean_absolute_error(y_test, tactical_pred):.4f}")

---
## Comparison

In [ ]:
naive_r2 = r2_score(y_test, naive_pred)
tact_r2 = r2_score(y_test, tactical_pred)
naive_mae = mean_absolute_error(y_test, naive_pred)
tact_mae = mean_absolute_error(y_test, tactical_pred)

comp = pd.DataFrame({
    "Model": ["Naive", "Tactical"],
    "R2_test": [naive_r2, tact_r2],
    "MAE_test": [naive_mae, tact_mae],
    "R2_adj_train": [naive.rsquared_adj, tactical.rsquared_adj],
})
print(comp.to_string(index=False, float_format="{:.4f}".format))
print(f"\nR² gain: {tact_r2 - naive_r2:+.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, name, pred in [(axes[0], "Naive", naive_pred), (axes[1], "Tactical", tactical_pred)]:
    ax.scatter(y_test, pred, alpha=0.15, s=10)
    lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    ax.plot(lims, lims, "r--", linewidth=0.5)
    ax.set_xlabel("Actual delta")
    ax.set_ylabel("Predicted delta")
    ax.set_title(f"{name} — Actual vs Predicted")
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

rice = mf[mf['wy_player_id'] == 379209].iloc[0]

# Team tactical context
print("=" * 60)
print("  TEAM TACTICAL QUALITIES (z-scores)")
print("=" * 60)
header = "  {:<12} {:>10} {:>10} {:>10}".format("Quality", "West Ham", "Arsenal", "Delta")
print(header)
print("-" * 60)
for tq in ["ATTACK", "DEFENCE", "OUTCOME"]:
    fr = rice[f"from_q_proj_{tq}"]
    to = rice[f"to_q_{tq}"]
    print("  {:<12} {:>10.3f} {:>10.3f} {:>+10.3f}".format(tq, fr, to, to - fr))
print()

# Passing quality
print("=" * 60)
print("  PASSING QUALITY")
print("=" * 60)
pre  = rice["from_Passing quality"]
post = rice["to_Passing quality"]
delta_actual = post - pre
print(f"  Pre-transfer (West Ham):  {pre:.3f}")
print(f"  Post-transfer (Arsenal):  {post:.3f}  (actual)")
print(f"  Delta actual:            {delta_actual:+.3f}")
print()

# Model predictions
rice_feat_naive = sm.add_constant(pd.DataFrame({"from_Passing quality": [pre]}))
rice_feat_tact = sm.add_constant(pd.DataFrame({
    "from_Passing quality": [pre],
    "delta_tq_ATTACK": [rice["delta_tq_ATTACK"]],
    "delta_tq_DEFENCE": [rice["delta_tq_DEFENCE"]],
    "delta_tq_OUTCOME": [rice["delta_tq_OUTCOME"]],
}))

pred_naive = naive.predict(rice_feat_naive)[0]
pred_tact  = tactical.predict(rice_feat_tact)[0]

print("=" * 60)
print("  PREDICTIONS")
print("=" * 60)
print(f"  Naive predicted delta:    {pred_naive:+.3f}  ->  to_PQ = {pre + pred_naive:.3f}")
print(f"  Tactical predicted delta: {pred_tact:+.3f}  ->  to_PQ = {pre + pred_tact:.3f}")
print(f"  Actual:                   {delta_actual:+.3f}  ->  to_PQ = {post:.3f}")
print()
print(f"  Naive error:    {abs(delta_actual - pred_naive):.3f}")
print(f"  Tactical error: {abs(delta_actual - pred_tact):.3f}")


In [ ]:
rice = mf[mf['wy_player_id'] == 379209].iloc[0]

# Team tactical context
print("═" * 60)
print("  TEAM TACTICAL QUALITIES (z-scores)")
print("═" * 60)
print(f"  {"Quality":<12} {"West Ham":>10} {"Arsenal":>10} {"Delta":>10}")
print("─" * 60)
for tq in ["ATTACK", "DEFENCE", "OUTCOME"]:
    fr = rice[f"from_q_proj_{tq}"]
    to = rice[f"to_q_{tq}"]
    print(f"  {tq:<12} {fr:>10.3f} {to:>10.3f} {to - fr:>+10.3f}")
print()

# Passing quality
print("═" * 60)
print("  PASSING QUALITY")
print("═" * 60)
pre  = rice["from_Passing quality"]
post = rice["to_Passing quality"]
delta_actual = post - pre
print(f"  Pre-transfer (West Ham):  {pre:.3f}")
print(f"  Post-transfer (Arsenal):  {post:.3f}  (actual)")
print(f"  Delta actual:            {delta_actual:+.3f}")
print()

# Model predictions
rice_feat_naive = sm.add_constant(pd.DataFrame({"from_Passing quality": [pre]}))
rice_feat_tact = sm.add_constant(pd.DataFrame({
    "from_Passing quality": [pre],
    "delta_tq_ATTACK": [rice["delta_tq_ATTACK"]],
    "delta_tq_DEFENCE": [rice["delta_tq_DEFENCE"]],
    "delta_tq_OUTCOME": [rice["delta_tq_OUTCOME"]],
}))

pred_naive = naive.predict(rice_feat_naive)[0]
pred_tact  = tactical.predict(rice_feat_tact)[0]

print("═" * 60)
print("  PREDICTIONS")
print("═" * 60)
print(f"  Naive predicted delta:    {pred_naive:+.3f}  →  to_PQ = {pre + pred_naive:.3f}")
print(f"  Tactical predicted delta: {pred_tact:+.3f}  →  to_PQ = {pre + pred_tact:.3f}")
print(f"  Actual:                   {delta_actual:+.3f}  →  to_PQ = {post:.3f}")
print()
print(f"  Naive error:    {abs(delta_actual - pred_naive):.3f}")
print(f"  Tactical error: {abs(delta_actual - pred_tact):.3f}")